### LangGraph test

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph

# ---- 1. 상태(state) 정의 ---
# 그래프 전체에서 공유될 데이터 구조를 정의합니다.
# 'value'라는 키에 정수(int) 값이 저장됩니다.

class GraphState(TypedDict) :
  """
  그래프의 상태를 나타냅니다.
  
  Args:
    value (int) : 0에서 시작하여 1씩 증가할 카운터 값
  """
  value: int
  
# --- 2. 노드 (Node) 함수 정의 ---
# 각 노드는 현재 상태(state)를 입력으로 받고,
# 상태를 어떻게 변경할지(또는 변경하지 않을지) 딕셔너리 형태로 반환합니다.

def add_one(state: GraphState) :
  """
  현재 상태의 'value' 값에 1을 더합니다.
  """
  
  # 현재 상태를 읽어 옵니다.
  current_value = state['value']
  print(f"--- 'add_one' 노드 실행")
  print(f".   ( 입력 ) 현재 상태 'value': {current_value}")
  
  # 상태를 변경합니다.
  new_value = current_value + 1
  
  print(f".   ( 출력 ) 'value'를 {new_value}로 업데이트")
  
  # 변경할 상태만 딕셔너리로 반환합니다.
  return {"value": new_value}

# --- 3. 조건부 엣지(Edge) 함수 정의 ---
# 이 함수는 현재 상태를 *읽고* 다음에 어떤 노드로 가야 할지 결정합니다.

def should_continue(state: GraphState) : 
    """
    상태의 'value' 값을 확인하여 다음 단계를 결정합니다.
    """
    
    current_value = state['value']
    print(f"--- 'should_continue' 조건부 엣지 실행 ---")
    print(f".   ( 상태확인 ) 현재  'value': {current_value}")
    
    if current_value < 3:
        # 값이 3보다 작으면 'continue_adding' 경로를 반환
        print(f"  (결정) 값이 3보다 작으므로 'add one' 노드로 다시 이동")
        return "continue_adding"
    else:
      # 값이 3이상이면 'end_graph" 경로를 반환
      print(f".   ( 결정 ) 값이 3 이상이므로 그래프 종료")
      return "end_graph"
    
  # --- 4. 그래프 생성 및 노드//엣지 연결 ---
  
  #GraphState 를 사용하는 StateGraph 객체 생성
  
workflow = StateGraph(GraphState)
  
  # 노드를 그래프에 추가 (이름, 실행할 함수)
workflow.add_node('adder_node', add_one)
  
  # 그래프의 시작점을 'adder_node'로 설정
workflow.set_entry_point('adder_node')
  
  # 조건부 엣지(간선) 추가
  # 'adder_node' 가 실행된 후, 'should_continue' 함수를 호출하여 상태를 확인
  
workflow.add_conditional_edges(
    'adder_node', #시작노드
    should_continue, # 상태를 확인할 함수
    {
      # 'should continue'가 'continue_adding'을 반환하면 -> 'adder_node'로 이동 (루프)
      "continue_adding": "adder_node",
      
      # 'should continue'가 'end_graph'을 반환하면 -> 그래프 종료
      "end_graph": "__end__"
    }
  )
  
  # 그래프를 실행 가능한 객체로 컴파일
  
app = workflow.compile()
  
print("--- 그래프 실행 시작 (초기상태: {'value':0}) ---")
  
# invoke : 그래프를 실행하고 최종 상태를 반환
# {'value': 0 }으로 초기 상태를 설정하여 실행
final_state = app.invoke({"value": 0})

print("\n--- 그래프 실행 종료 ---")
print(f"최종상태: {final_state}")
  

--- 그래프 실행 시작 (초기상태: {'value':0}) ---
--- 'add_one' 노드 실행
.   ( 입력 ) 현재 상태 'value': 0
.   ( 출력 ) 'value'를 1로 업데이트
--- 'should_continue' 조건부 엣지 실행 ---
.   ( 상태확인 ) 현재  'value': 1
  (결정) 값이 3보다 작으므로 'add one' 노드로 다시 이동
--- 'add_one' 노드 실행
.   ( 입력 ) 현재 상태 'value': 1
.   ( 출력 ) 'value'를 2로 업데이트
--- 'should_continue' 조건부 엣지 실행 ---
.   ( 상태확인 ) 현재  'value': 2
  (결정) 값이 3보다 작으므로 'add one' 노드로 다시 이동
--- 'add_one' 노드 실행
.   ( 입력 ) 현재 상태 'value': 2
.   ( 출력 ) 'value'를 3로 업데이트
--- 'should_continue' 조건부 엣지 실행 ---
.   ( 상태확인 ) 현재  'value': 3
.   ( 결정 ) 값이 3 이상이므로 그래프 종료

--- 그래프 실행 종료 ---
최종상태: {'value': 3}


### 랭그래프 기본 뼈대 코드

In [3]:
import os
import json
import asyncio
from typing import TypedDict, List
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_postgres import PGVector
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END


/Users/hyunjunson/Project/green-hat/backend/ml-pipline/yes/envs/oracle-langchain/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- 환경 설정 ---


load_dotenv()

# ============================================================
# 1단계: 문서 로드
# ============================================================
print("\n[1단계] 문서 로드 (Document Loader)")
print("-" * 80)

FILE_PATH = "data/SPRi AI Brief_10월호_산업동향_1002_F.pdf"
loader = PyPDFLoader(FILE_PATH)
documents = loader.load()

print(f"총 {len(documents)} 페이지 로드 완료")
print(f"첫 페이지 미리보기: {documents[0].page_content[:100]}...")

# ============================================================
# 2단계: 텍스트 분할
# ============================================================
print("\n[2단계] 텍스트 분할 (Text Splitter)")
print("-" * 80)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
)
splits = text_splitter.split_documents(documents)

print(f"총 {len(splits)}개의 청크로 분할 완료")
print(f"첫 번째 청크: {splits[0].page_content[:150]}...")

# ============================================================
# 3단계: 임베딩
# ============================================================
print("\n[3단계] 임베딩 (Embedding)")
print("-" * 80)

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-large")
sample_text = "생성형 AI 기술 동향"
sample_vector = embeddings_model.embed_query(sample_text)

print("임베딩 모델 준비 완료")
print("모델: text-embedding-3-large (OpenAI)")
print(f"벡터 차원: {len(sample_vector)}차원")
print(f"샘플 벡터 (처음 5개): {sample_vector[:5]}")

# ============================================================
# DB 연결 설정
# ============================================================
DB_CONFIG = {
    'host': os.getenv("DB_HOST"),
    'port': os.getenv("DB_PORT"),
    'database': os.getenv("DB_NAME"),
    'user': os.getenv("DB_USER"),
    'password': os.getenv("DB_PASS")
}

# Vector store 생성
CONNECTION_STRING = PGVector.connection_string_from_db_params(driver="psycopg", **DB_CONFIG)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-large")

vectorstore = PGVector.from_documents(
    documents=documents,
    embedding=embeddings_model,
    collection_name="langgraph_test",
    connection=CONNECTION_STRING,   
    pre_delete_collection=True
)

# Vector retriever 생성
vector_retriever = vectorstore.as_retriever(search_type="similarity",search_kwargs={"k": 3})
# # ============================================================
# # 4단계: Web Search 도구 정의
# # ============================================================
@tool
def web_search(query: str) -> str:
    """웹 검색 수행 (Google Serper API 사용)"""
    print('-------- WEB SEARCH --------')
    print(f"Query: {query}")

    search = GoogleSerperAPIWrapper()
    result = search.run(query)
    return result

# ============================================================
# Part 2: LangGraph 설계
# ============================================================
print("\n[Part 2] LangGraph 설계")
print("-" * 80)

# ---------- 그래프 상태 정의 ----------
class GraphState(TypedDict):
    question: str
    documents: List[Document]
    generation: str
    decision: str  # 'yes', 'no', 'web_search'

# ---------- LLM ----------
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# ---------- Pydantic 모델 ----------
class Grader(BaseModel):
    """문서의 관련성을 평가하기 위한 모델"""
    decision: str = Field(description="결정은 'yes', 'no', 'web_search' 중 하나여야 합니다.")

# ---------- 노드 정의 ----------
def retrieve(state: GraphState):
    print(" --- 노드 실행: retrieve ---")
    question = state['question']
    retriever = vectorstore.as_retriever()
    documents = retriever.invoke(question)
    return {"documents": documents}

def grade_documents(state: GraphState):
    print("--- 노드 실행: grade_documents ---")
    question = state['question']
    documents = state['documents']

    prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 문서 평가 전문가입니다.
        문서가 질문에 답변하기 충분한지 다음 중 하나로 판단하세요:
        - 'yes': 문서만으로 충분함
        - 'web_search': 관련은 있으나 보강 필요
        - 'no': 전혀 관련 없음
        질문: {question}
        문서 내용: {documents}""")
    ])

    structured_llm_grader = llm.with_structured_output(Grader)
    doc_str = "\n\n".join(doc.page_content for doc in documents)
    chain = prompt | structured_llm_grader
    response = chain.invoke({"question": question, "documents": doc_str})

    print(f"문서 평가 결과: {response.decision}")
    return {"decision": response.decision}

def generate(state: GraphState):
    print("--- 노드 실행: generate ---")
    question = state['question']
    documents = state['documents']

    prompt = ChatPromptTemplate.from_template("""
    주어진 문맥 정보를 사용하여 다음 질문에 답변하세요.
    질문: {question}
    문맥:
    {context}
    """)

    context = "\n\n".join(f"[출처: {doc.metadata.get('source', '문서')}]\n{doc.page_content}" for doc in documents)
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": question, "context": context})
    return {"generation": answer}

def transform_query(state: GraphState):
    print("--- 노드 실행: transform_query ---")
    question = state["question"]

    prompt = ChatPromptTemplate.from_template("""
    사용자의 질문을 웹 검색에 더 적합한 검색어로 재구성하세요.
    재구성된 검색어만 반환하세요.
    원래 질문: {question}
    """)

    chain = prompt | llm | StrOutputParser()
    better_question = chain.invoke({"question": question})
    print(f"재구성된 질문: {better_question}")
    return {"question": better_question}

def web_search_node(state: GraphState):
    print("--- 노드 실행: web_search_node ---")
    question = state['question']
    search_result = web_search.invoke(question)
    web_docs = [Document(page_content=search_result, metadata={"source": "web_search"})]
    return {"documents": web_docs}

def web_search_for_supplement(state: GraphState):
    print("--- 노드 실행: web_search_for_supplement ---")
    question = state['question']
    existing_docs = state["documents"]

    search_result = web_search.invoke(question)
    web_docs = [Document(page_content=search_result, metadata={"source": "web_search"})]
    combined = existing_docs + web_docs
    print(f"기존 문서 {len(existing_docs)}개 + 웹 검색 결과 {len(web_docs)}개 통합")
    return {"documents": combined}

# ---------- 조건부 엣지 ----------
def route_after_grading(state: GraphState) -> str:
    print(" --- 조건부 엣지 실행: route_after_grading ---")
    decision = state["decision"]
    if decision == "yes":
        print("결정: 문서 충분 → generate")
        return "generate"
    elif decision == "no":
        print("결정: 문서 무관 → transform_query")
        return "transform_query"
    elif decision == "web_search":
        print("결정: 보충 필요 → web_search_for_supplement")
        return "web_search_for_supplement"

# ============================================================
# 그래프 구성
# ============================================================
workflow = StateGraph(GraphState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("transform_query", transform_query)
workflow.add_node("web_search_node", web_search_node)
workflow.add_node("web_search_for_supplement", web_search_for_supplement)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")

workflow.add_conditional_edges(
    "grade_documents",
    route_after_grading,
    {
        "generate": "generate",
        "transform_query": "transform_query",
        "web_search_for_supplement": "web_search_for_supplement",
    },
)

workflow.add_edge("transform_query", "web_search_node")
workflow.add_edge("web_search_node", "generate")
workflow.add_edge("web_search_for_supplement", "generate")

app = workflow.compile()
print("\nCRAG 그래프 컴파일 완료!")

# ============================================================
# 그래프 시각화
# ============================================================
try:
    img_bytes = app.get_graph().draw_mermaid_png()
    with open("crag_graph.png", "wb") as f:
        f.write(img_bytes)
    print("그래프 구조가 'crag_graph.png'로 저장되었습니다.")
except Exception as e:
    print(f"그래프 시각화 실패 (Graphviz 필요): {e}")

# ============================================================
# 실행부
# ============================================================
async def main():
    print("\n" + "=" * 90)
    print("Corrective RAG 시스템을 시작합니다.")
    print("종료하려면 'quit' 또는 'exit'를 입력하세요.")
    print("=" * 90)

    while True:
        user_question = input("\nYou: ")
        if user_question.lower() in ['quit', 'exit', '종료']:
            print("\n시스템을 종료합니다.")
            break

        if not user_question.strip():
            continue

        inputs = {"question": user_question}

        try:
            print("\nAI: ", end="", flush=True)
            current_generating = False
            async for event in app.astream_events(inputs, version="v2"):
                kind = event['event']

                if kind == "on_chain_start" and event["name"] == 'generate':
                    current_generating = True
                elif kind == "on_chain_end" and event["name"] == "generate":
                    current_generating = False
                elif kind == "on_chat_model_stream" and current_generating:
                    content = event["data"]["chunk"].content
                    if content:
                        print(content, end="", flush=True)

            print("\n" + "-" * 80)
        except Exception as e:
            print(f"Error: {e}")


[1단계] 문서 로드 (Document Loader)
--------------------------------------------------------------------------------
총 29 페이지 로드 완료
첫 페이지 미리보기: 2025년10월호인공지능 산업의 최신 동향...

[2단계] 텍스트 분할 (Text Splitter)
--------------------------------------------------------------------------------
총 70개의 청크로 분할 완료
첫 번째 청크: 2025년10월호인공지능 산업의 최신 동향...

[3단계] 임베딩 (Embedding)
--------------------------------------------------------------------------------
임베딩 모델 준비 완료
모델: text-embedding-3-large (OpenAI)
벡터 차원: 3072차원
샘플 벡터 (처음 5개): [-0.007897513918578625, -0.04372946918010712, -0.007173288147896528, -0.026099732145667076, 0.02313385345041752]

[Part 2] LangGraph 설계
--------------------------------------------------------------------------------

CRAG 그래프 컴파일 완료!
그래프 구조가 'crag_graph.png'로 저장되었습니다.
